In [1]:
from zipfile import ZipFile
file_name = "dataset.zip"

with ZipFile(file_name, 'r') as zip:
  zip.extractall()
  print('Done')

Done


In [2]:
!pip install gradio huggingface_hub

In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
code = """
import pandas as pd
import numpy as np
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
import gradio as gr

# Load book metadata
books = pd.read_csv("books_with_emotions.csv")
books["large_thumbnail"] = books["thumbnail"] + "&fife=w800"
books["large_thumbnail"] = np.where(
    books["large_thumbnail"].isna(),
    "cover-not-found.jpg",
    books["large_thumbnail"],
)

# Load Chroma vector DB
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db_books = Chroma(
    collection_name="book_collection",
    embedding_function=embedding_model,
    persist_directory="chroma_db_books"
)

# Recommendation logic
def retrieve_semantic_recommendations(query, category=None, tone=None, initial_top_k=50, final_top_k=16):
    recs = db_books.similarity_search(query, k=initial_top_k)
    books_list = [int(rec.page_content.strip('"').split()[0]) for rec in recs]
    book_recs = books[books["isbn13"].isin(books_list)].head(initial_top_k)

    if category != "All":
        book_recs = book_recs[book_recs["simple_categories"] == category].head(final_top_k)
    else:
        book_recs = book_recs.head(final_top_k)

    if tone == "Happy":
        book_recs.sort_values(by="joy", ascending=False, inplace=True)
    elif tone == "Surprising":
        book_recs.sort_values(by="surprise", ascending=False, inplace=True)
    elif tone == "Angry":
        book_recs.sort_values(by="anger", ascending=False, inplace=True)
    elif tone == "Suspenseful":
        book_recs.sort_values(by="fear", ascending=False, inplace=True)
    elif tone == "Sad":
        book_recs.sort_values(by="sadness", ascending=False, inplace=True)

    return book_recs

def recommend_books(query, category, tone):
    recommendations = retrieve_semantic_recommendations(query, category, tone)
    results = []

    for _, row in recommendations.iterrows():
        description = row["description"]
        truncated_desc_split = description.split()
        truncated_description = " ".join(truncated_desc_split[:30]) + "..."

        authors_split = row["authors"].split(";")
        if len(authors_split) == 2:
            authors_str = f"{authors_split[0]} and {authors_split[1]}"
        elif len(authors_split) > 2:
            authors_str = f"{', '.join(authors_split[:-1])}, and {authors_split[-1]}"
        else:
            authors_str = row["authors"]

        caption = f"{row['title']} by {authors_str}: {truncated_description}"
        results.append((row["large_thumbnail"], caption))
    return results

# Gradio UI
categories = ["All"] + sorted(books["simple_categories"].unique())
tones = ["All", "Happy", "Surprising", "Angry", "Suspenseful", "Sad"]

with gr.Blocks(theme=gr.themes.Glass()) as dashboard:
    gr.Markdown("# AI-Powered Book Discovery Platform")

    with gr.Row():
        user_query = gr.Textbox(label="Please enter a description of a book:", placeholder="e.g., A story about forgiveness")
        category_dropdown = gr.Dropdown(choices=categories, label="Select a category:", value="All")
        tone_dropdown = gr.Dropdown(choices=tones, label="Select an emotional tone:", value="All")
        submit_button = gr.Button("Find recommendations")

    gr.Markdown("## Recommendations")
    output = gr.Gallery(label="Recommended books", columns=4, rows=2)

    submit_button.click(fn=recommend_books, inputs=[user_query, category_dropdown, tone_dropdown], outputs=output)

dashboard.launch()
"""
with open("app.py", "w") as f:
    f.write(code)

In [5]:
reqs = """
gradio
langchain
langchain-community
chromadb
sentence-transformers
pandas
numpy
python-dotenv
"""
with open("requirements.txt", "w") as f:
    f.write(reqs)

In [6]:

from huggingface_hub import notebook_login
notebook_login()

In [8]:
from huggingface_hub import create_repo

repo_id = "Danielj08/AI-Powered-Book-Discovery-Platform"
create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepoUrl('https://huggingface.co/spaces/Danielj08/AI-Powered-Book-Discovery-Platform', endpoint='https://huggingface.co', repo_type='space', repo_id='Danielj08/AI-Powered-Book-Discovery-Platform')

In [9]:
from huggingface_hub import HfApi

api = HfApi()

# Upload individual files
api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=repo_id, repo_type="space")
api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=repo_id, repo_type="space")
api.upload_file(path_or_fileobj="books_with_emotions.csv", path_in_repo="books_with_emotions.csv", repo_id=repo_id, repo_type="space")
api.upload_file(path_or_fileobj="cover-not-found.jpg", path_in_repo="cover-not-found.jpg", repo_id=repo_id, repo_type="space")

# Upload chroma DB folder
api.upload_folder(folder_path="chroma_db_books", path_in_repo="chroma_db_books", repo_id=repo_id, repo_type="space")

Uploading...:   0%|          | 0.00/115M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/spaces/Danielj08/AI-Powered-Book-Discovery-Platform/commit/d6b58b7dd6b67957001df75ca9ca865614b4094e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d6b58b7dd6b67957001df75ca9ca865614b4094e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/Danielj08/AI-Powered-Book-Discovery-Platform', endpoint='https://huggingface.co', repo_type='space', repo_id='Danielj08/AI-Powered-Book-Discovery-Platform'), pr_revision=None, pr_num=None)